# Match Events Analysis

Summaries and visuals for `match_events` and `players` from `events_etl.py`.

In [ ]:
import duckdb, glob, pandas as pd, numpy as np, plotly.express as px

In [ ]:
db_candidates = sorted(glob.glob('../../data/mydb2024-25*.duckdb'))
db_path = db_candidates[-1] if db_candidates else '../data/mydb.duckdb'
print(f'Using DuckDB database: {db_path}')
con = duckdb.connect(db_path)

In [ ]:
# Load tables (if present)
try:
    df_ev = con.execute('SELECT * FROM match_events').df()
except Exception:
    df_ev = pd.DataFrame()
try:
    df_players = con.execute('SELECT * FROM players').df()
except Exception:
    df_players = pd.DataFrame()
len(df_ev), len(df_players)

In [ ]:
# Event types count
if not df_ev.empty and 'eventType' in df_ev.columns:
    display(df_ev['eventType'].value_counts().rename_axis('eventType').reset_index(name='count').head(50))
else:
    print('No match_events or eventType column missing.')

In [ ]:
# Shot map (if x,y available)
shot_mask = (df_ev.get('x').notna() & df_ev.get('y').notna()) if not df_ev.empty else pd.Series([], dtype=bool)
df_shots = df_ev[shot_mask] if not df_ev.empty else pd.DataFrame()
if not df_shots.empty:
    color_col = 'success' if 'success' in df_shots.columns else None
    px.scatter(df_shots, x='x', y='y', color=color_col, title='Events (x,y)', opacity=0.6)
else:
    print('No x,y positions available.')

In [ ]:
# Top scorers by personName
if not df_ev.empty and 'personName' in df_ev.columns and 'success' in df_ev.columns:
    goals = df_ev[df_ev['success'] == True].groupby('personName').size().sort_values(ascending=False).head(20)
    px.bar(goals.reset_index().rename(columns={0:'goals'}), x='personName', y='goals', title='Top Scorers (success=True)')
else:
    print('personName/success not available.')